# Coffee spend by age group and gender: category scores

**Hayashi's Quantification Method I** · a worked example from the StatNex teaching guide

**The story.** A cafe looked at 13 regular customers' monthly coffee spending, together with their age group and gender, which are both categories rather than numbers.

**The question.** Which matters more for coffee spending: age group or gender?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/quantification_i.csv` (13 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Coffee spend ($/month),Age group,Gender
1.0,18.0,Teens,Female
2.0,25.0,Teens,Male
3.0,22.0,Teens,Female
4.0,45.0,20s,Male
5.0,60.0,20s,Female
6.0,52.0,20s,Male
7.0,75.0,20s,Female
8.0,38.0,20s,Male
9.0,70.0,30s,Male
10.0,55.0,30s,Female
11.0,82.0,30s,Male
12.0,40.0,40s,Male
13.0,48.0,40s,Female

These cells save `categories.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile categories.py
"""The categories of a qualitative variable, in the order StatNex lists them.

Each value is written as the sheet shows it (a code read as 2.0 is "2"). When every
category is a number they go from the smallest up: 1, 2, 3, 10, where sorting them as
text would put 10 between 1 and 2. Otherwise they are sorted by their bytes in
Shift_JIS (CP932) -- the order a Japanese Excel sorts by, which for kanji runs by
reading. For plain English labels this is ordinary alphabetical order with capitals
first. pandas has no such sort, so it is written out here.
"""

import numpy as np


def as_text(value):
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def cp932_key(text):
    try:
        return text.encode("cp932")
    except UnicodeEncodeError:
        return text.encode("utf-8")


def category_order(labels):
    """The distinct labels in order: numbers from the smallest up, else by CP932 bytes."""
    labels = set(labels)
    try:
        return sorted(labels, key=float)
    except ValueError:
        return sorted(labels, key=cp932_key)


def categories(column):
    """(the categories in order, each case's category as 0, 1, 2, ...)."""
    text = column.map(as_text)
    ordered = category_order(text)
    codes = text.map({name: i for i, name in enumerate(ordered)}).to_numpy(int)
    return ordered, codes


def dummies(codes, k):
    """One 0/1 column per category: 1 where the case falls in that category."""
    return np.eye(k)[codes]

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from categories import categories, dummies

pd.set_option("display.width", 120)

## The data

In [ ]:
# The response is a number; each explanatory item is a set of categories. A case with
# a blank in any of them is left out, and text in the response counts as blank.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Coffee spend ($/month)"
items = ["Age group", "Gender"]
y_all = pd.to_numeric(data[response], errors="coerce")
blank = data[items].isna().any(axis=1)
valid = y_all.notna() & ~blank
summary = pd.DataFrame(
    {"n": [valid.sum(), (y_all.isna() & ~blank).sum(), (y_all.notna() & blank).sum(),
           (y_all.isna() & blank).sum(), len(data)]},
    index=["Valid cases", "Response missing only", "Items missing only", "Both missing", "Total"],
)
summary["share"] = summary["n"] / len(data)
print(summary)

y = y_all[valid].to_numpy(float)
n = len(y)
# Each item's categories, sorted as StatNex lists them (categories.py).
levels, codes = {}, {}
for item in items:
    levels[item], codes[item] = categories(data.loc[valid, item])

## The response in each category

In [ ]:
rows = []
for item in items:
    for k, category in enumerate(levels[item]):
        group = y[codes[item] == k]
        rows.append([item, category, len(group), group.mean(),
                     group.var(ddof=1) if len(group) > 1 else np.nan,
                     group.std(ddof=1) if len(group) > 1 else np.nan])
rows.append(["Total", "", n, y.mean(), y.var(ddof=1), y.std(ddof=1)])
describe = pd.DataFrame(rows, columns=["item", "category", "n", "mean", "variance", "SD"])
print()
print(describe.to_string(index=False))

## Cross-tabulation of the categories

In [ ]:
# One 0/1 column per category. Every category against every other: an item against
# itself is just its counts on the diagonal.
design = np.column_stack([dummies(codes[item], len(levels[item])) for item in items])
labels = [f"{item}: {category}" for item in items for category in levels[item]]
crosstab = pd.DataFrame((design.T @ design).astype(int), index=labels, columns=labels)
print()
print(crosstab)

## Category scores

In [ ]:
# Quantification method I is an ordinary least-squares regression on 0/1 "dummy"
# columns, one per category. Each item's last category is left out (it is implied by
# the others), so a coefficient is that category's difference from the left-out one.
# statsmodels' OLS fits it. Then each item's scores are shifted so that their average
# over the cases is 0. The shift goes into the constant, which becomes the mean of y.
x = np.column_stack([np.ones(n)] + [dummies(codes[item], len(levels[item]))[:, :-1]
                                   for item in items])
model = sm.OLS(y, x).fit()
coefficients = model.params
constant = coefficients[0]
scores, start = {}, 1
for item in items:
    k = len(levels[item])
    raw = np.append(coefficients[start:start + k - 1], 0.0)
    start += k - 1
    counts = np.bincount(codes[item], minlength=k)
    shift = (counts * raw).sum() / n
    scores[item] = raw - shift
    constant += shift
category_scores = pd.DataFrame(
    [[item, category, scores[item][k]] for item in items for k, category in enumerate(levels[item])]
    + [["Constant", "", constant]],
    columns=["item", "category", "score"],
)
print()
print(category_scores.to_string(index=False))

## The same model as an ordinary regression

In [ ]:
# The coefficients before centring, with their standard errors and 95% confidence
# intervals (t distribution on n minus the number of coefficients df). Each is a
# category's difference from its item's left-out category, with the other items held
# fixed. A wide interval means the category holds too few cases to pin its score
# down. The fitted values and R squared are the method's own.
dummy_labels = ["Constant"] + [f"{item}: {category} vs {levels[item][-1]}"
                               for item in items for category in levels[item][:-1]]
regression = pd.DataFrame({"coefficient": model.params, "SE": model.bse}, index=dummy_labels)
regression[["lower 95%", "upper 95%"]] = model.conf_int(0.05)
print()
print(regression.round(4))

## Does each item matter?

In [ ]:
# An F test per item: do its categories, taken together, add anything once the other
# items are in the model? f_test() tests that all of the item's coefficients are 0;
# the test has (categories - 1) and the residual degrees of freedom. StatNex prints no
# test for an item; this is the one to quote, beside the range.
item_tests, start = [], 1
for item in items:
    k = len(levels[item]) - 1
    restriction = np.zeros((k, x.shape[1]))
    restriction[np.arange(k), start + np.arange(k)] = 1.0
    start += k
    test = model.f_test(restriction)
    item_tests.append([float(np.squeeze(test.fvalue)), int(test.df_num), int(test.df_denom),
                       float(test.pvalue)])
print()
print(pd.DataFrame(item_tests, index=items, columns=["F", "df1", "df2", "P"]).round(4))

## Ranges

In [ ]:
# The spread of an item's scores: the larger, the more the item moves the prediction.
# It measures size, not certainty: a category with two or three cases can stretch it.
ranges = pd.Series({item: scores[item].max() - scores[item].min() for item in items})
print()
print(ranges)

## Correlations with the response

In [ ]:
# Each item becomes one number per case: the score of the case's category. The simple
# correlation is that number against the response, item by item. The partial
# correlation first takes the other items' scores out of both, so it shows what the
# item adds; it comes from the inverse (here the pseudo-inverse) of the correlation
# matrix.
applied = np.column_stack([scores[item][codes[item]] for item in items])
simple = [np.corrcoef(y, applied[:, j])[0, 1] for j in range(len(items))]
if len(items) == 1:
    partial = simple
else:
    inverse = np.linalg.pinv(np.corrcoef(np.column_stack([y, applied]), rowvar=False))
    partial = [-inverse[0, j] / np.sqrt(inverse[0, 0] * inverse[j, j])
               for j in range(1, len(items) + 1)]
correlations = pd.DataFrame({"simple": simple, "partial": partial}, index=items)
print()
print(correlations)

## Accuracy

In [ ]:
# R squared is the share of the response's variation that the fitted values account
# for (model.rsquared above gives the same number), and the multiple R is its square
# root. Every extra category raises it a little even by chance, so with few cases per
# coefficient also read the adjusted R squared, printed after it.
fitted = constant + applied.sum(axis=1)
residual = y - fitted
r2 = 1 - (residual @ residual) / ((y - y.mean()) ** 2).sum()
accuracy = pd.Series({"multiple R": np.sqrt(max(r2, 0.0)), "R squared": r2})
print()
print(accuracy)
print(f"Adjusted R squared: {model.rsquared_adj:.4f}")

## Correlations between the items

In [ ]:
item_correlations = pd.DataFrame(np.corrcoef(applied, rowvar=False).reshape(len(items), len(items)),
                                 index=items, columns=items)
print()
print(item_correlations)

## Predicted values

In [ ]:
predictions = pd.DataFrame({"observed": y, "predicted": fitted, "residual": residual},
                           index=range(1, n + 1))
print()
print(predictions)

## What the result says

Quantification method I is a regression for predictors that are categories. Each category gets a score: the amount it adds to, or takes from, the average spend of \$48.46 a month.

For age group the scores run from -27.7 for teens to 21.1 for customers in their 30s. The range, the gap between the highest and the lowest score of an item, is 48.8. For gender the scores are 2.4 for women and -2.1 for men, a range of only 4.5. The partial correlation, the link with spending after allowing for the other item, tells the same story: 0.87 for age group and 0.22 for gender. The squared multiple correlation is 0.757, so the two items together account for about three quarters of the differences in spending.

So age group matters far more than gender. But with 13 customers, some age groups hold only 2 or 3 people, so the finer details are uncertain.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.